# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.27.2 — FAST
## Nested Angular Cubic Motion: line → circle → helix

### Conceptual correction

The nested cubes are **not static boxes placed inside one another**.

Each level may have a time-dependent relative motion:

\[
\mathbf x_{k+1}(t)
=
\mathbf T_{k+1\leftarrow k}(t)
+
R_{k+1\leftarrow k}(t)\mathbf x_k(t),
\]

with

\[
R_{k+1\leftarrow k}(t)\in SO(3),
\qquad
\boldsymbol\omega_{k+1\leftarrow k}(t)\neq0.
\]

This notebook constructs one explicit kinematic witness:

\[
\boxed{
\text{line in }C_0
\rightarrow
\text{circle in }C_1
\rightarrow
\text{helix in }C_2
}
\]

while testing inverse reconstruction of the same local trajectory.

This is a **prescribed kinematic construction** only. No force law, inter-scale dynamics, or new GVH physics is derived.

In [1]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd

UPSTREAM = {
    "p33271_canonical_sha256":"d17cfd913e079495bc3c421e4c3929029e9d007d3a4711f09eb3f0674ecad418",
    "G271_MINIMAL_NUMERICAL_GATE_PASS":True,
    "G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED":False,
    "G271_NEW_GVH_PHYSICS_VALIDATED":False,
}

UPSTREAM_GATE = (
    UPSTREAM["G271_MINIMAL_NUMERICAL_GATE_PASS"]
    and not UPSTREAM["G271_PHYSICAL_COARSE_GRAINING_LAW_DERIVED"]
    and not UPSTREAM["G271_NEW_GVH_PHYSICS_VALIDATED"]
)
assert UPSTREAM_GATE

print("UPSTREAM_GATE =",UPSTREAM_GATE)
print("P33271_CANONICAL_SHA256 =",UPSTREAM["p33271_canonical_sha256"])

UPSTREAM_GATE = True
P33271_CANONICAL_SHA256 = d17cfd913e079495bc3c421e4c3929029e9d007d3a4711f09eb3f0674ecad418


## 1 — Time-dependent rigid transform

For one level:

\[
\boxed{\mathbf x_P=\mathbf T(t)+R(t)\mathbf x_C}
\]

where \(C\) is the child cube and \(P\) its parent. The inverse is:

\[
\boxed{\mathbf x_C=R^T(t)[\mathbf x_P-\mathbf T(t)]}.
\]

For a rotation about \(z\):

\[
R_z(\theta)=
\begin{pmatrix}
\cos\theta&-\sin\theta&0\\
\sin\theta&\cos\theta&0\\
0&0&1
\end{pmatrix}.
\]

In [2]:
def Rz(theta):
    c,s=np.cos(theta),np.sin(theta)
    return np.array([[c,-s,0.0],[s,c,0.0],[0.0,0.0,1.0]])

def transform(T,R,x):
    return T + R @ x

def inverse_transform(T,R,x_parent):
    return R.T @ (x_parent-T)

# Representative SO(3) audit over time.
times_test=np.linspace(0.0,10.0,101)
errs=[]
for tt in times_test:
    R=Rz(0.7*tt)
    errs.append(np.max(np.abs(R.T@R-np.eye(3))))
SO3_TIME_DEPENDENT_ROTATION_PASS=max(errs)<1e-14
assert SO3_TIME_DEPENDENT_ROTATION_PASS

print("max_SO3_error =",max(errs))
print("SO3_TIME_DEPENDENT_ROTATION_PASS =",SO3_TIME_DEPENDENT_ROTATION_PASS)

max_SO3_error = 2.220446049250313e-16
SO3_TIME_DEPENDENT_ROTATION_PASS = True


## 2 — Level \(C_0\): rectilinear body trajectory

Choose

\[
\mathbf x_0(t)=
\begin{pmatrix}
x_{00}+v_0t\\0\\0
\end{pmatrix}.
\]

Thus the body is exactly rectilinear in its local cube \(C_0\). Benchmark units are dimensionless.

In [3]:
Tmax=20.0
N=4001
t=np.linspace(0.0,Tmax,N)

x00=1.25
v0=0.18
R_circle=7.0
omega10=0.45
omega21=0.20
u2=0.35

x0=np.column_stack([x00+v0*t,np.zeros_like(t),np.zeros_like(t)])
line_second_diff=np.max(np.abs(np.diff(x0,n=2,axis=0)))
C0_RECTILINEAR_PASS=line_second_diff<1e-12
assert C0_RECTILINEAR_PASS

print("C0 start =",x0[0])
print("C0 end   =",x0[-1])
print("max_line_second_difference =",line_second_diff)
print("C0_RECTILINEAR_PASS =",C0_RECTILINEAR_PASS)

C0 start = [1.25 0.   0.  ]
C0 end   = [4.85 0.   0.  ]
max_line_second_difference = 1.7763568394002505e-15
C0_RECTILINEAR_PASS = True


## 3 — Level \(C_1\): rotating child cube and exact circular parent trajectory

A generic line inside a rotating parent does **not automatically** become an exact circle.

To construct an exact witness, prescribe

\[
R_{1\leftarrow0}(t)=R_z(\omega_{10}t)
\]

and target

\[
\mathbf x_1(t)=
\begin{pmatrix}
R_c\cos(\omega_{10}t)\\
R_c\sin(\omega_{10}t)\\0
\end{pmatrix}.
\]

Then define the child-origin translation

\[
\mathbf T_{1\leftarrow0}(t)
=
\mathbf x_1^{\rm target}(t)
-
R_{1\leftarrow0}(t)\mathbf x_0(t).
\]

This is an explicit **existence witness**, not a derived dynamical law.

In [4]:
x1_target=np.column_stack([
    R_circle*np.cos(omega10*t),
    R_circle*np.sin(omega10*t),
    np.zeros_like(t)
])

x1=np.empty_like(x0)
T10=np.empty_like(x0)
for i,tt in enumerate(t):
    R10=Rz(omega10*tt)
    T10[i]=x1_target[i]-R10@x0[i]
    x1[i]=transform(T10[i],R10,x0[i])

circle_radius=np.sqrt(x1[:,0]**2+x1[:,1]**2)
C1_CIRCLE_RADIUS_PASS=np.max(np.abs(circle_radius-R_circle))<1e-12
C1_CIRCLE_Z_PASS=np.max(np.abs(x1[:,2]))<1e-12
C1_EXACT_CIRCLE_PASS=C1_CIRCLE_RADIUS_PASS and C1_CIRCLE_Z_PASS
assert C1_EXACT_CIRCLE_PASS

print("radius mean =",float(np.mean(circle_radius)))
print("max_radius_error =",float(np.max(np.abs(circle_radius-R_circle))))
print("C1_EXACT_CIRCLE_PASS =",C1_EXACT_CIRCLE_PASS)

radius mean = 7.0
max_radius_error = 1.7763568394002505e-15
C1_EXACT_CIRCLE_PASS = True


## 4 — Level \(C_2\): additional angular motion + axial transport

Let

\[
R_{2\leftarrow1}(t)=R_z(\omega_{21}t)
\]

and

\[
\mathbf T_{2\leftarrow1}(t)=
\begin{pmatrix}0\\0\\u_2t\end{pmatrix}.
\]

Then

\[
\mathbf x_2(t)
=
\mathbf T_{2\leftarrow1}(t)
+
R_{2\leftarrow1}(t)\mathbf x_1(t).
\]

Because \(\mathbf x_1\) is circular, the result is an exact helix:

\[
r_\perp=R_c,
\qquad z=u_2t,
\qquad \phi=(\omega_{10}+\omega_{21})t.
\]

In [5]:
x2=np.empty_like(x1)
T21=np.column_stack([np.zeros_like(t),np.zeros_like(t),u2*t])
for i,tt in enumerate(t):
    R21=Rz(omega21*tt)
    x2[i]=transform(T21[i],R21,x1[i])

rho2=np.sqrt(x2[:,0]**2+x2[:,1]**2)
phi2=np.unwrap(np.arctan2(x2[:,1],x2[:,0]))
coef_phi=np.polyfit(t,phi2,1)
coef_z=np.polyfit(t,x2[:,2],1)

C2_HELIX_RADIUS_PASS=np.max(np.abs(rho2-R_circle))<1e-12
C2_HELIX_Z_LINEAR_PASS=np.max(np.abs(x2[:,2]-u2*t))<1e-12
C2_HELIX_ANGULAR_RATE_PASS=abs(coef_phi[0]-(omega10+omega21))<1e-12
C2_EXACT_HELIX_PASS=all([C2_HELIX_RADIUS_PASS,C2_HELIX_Z_LINEAR_PASS,C2_HELIX_ANGULAR_RATE_PASS])
assert C2_EXACT_HELIX_PASS

print("helix radius mean =",float(np.mean(rho2)))
print("fitted angular rate =",coef_phi[0])
print("expected angular rate =",omega10+omega21)
print("fitted z rate =",coef_z[0])
print("expected z rate =",u2)
print("C2_EXACT_HELIX_PASS =",C2_EXACT_HELIX_PASS)

helix radius mean = 7.0
fitted angular rate = 0.6499999999999998
expected angular rate = 0.65
fitted z rate = 0.3499999999999998
expected z rate = 0.35
C2_EXACT_HELIX_PASS = True


## 5 — Inverse reconstruction: one kinematic history

Require

\[
\mathbf x_1
=
R_{2\leftarrow1}^T(\mathbf x_2-\mathbf T_{2\leftarrow1})
\]

and then

\[
\mathbf x_0
=
R_{1\leftarrow0}^T(\mathbf x_1-\mathbf T_{1\leftarrow0}).
\]

A successful inverse reconstruction shows that line/circle/helix are mutually consistent descriptions in nested moving frames, not three unrelated numerical trajectories.

In [6]:
x1_rec=np.empty_like(x1)
x0_rec=np.empty_like(x0)
for i,tt in enumerate(t):
    R21=Rz(omega21*tt)
    x1_rec[i]=inverse_transform(T21[i],R21,x2[i])
    R10=Rz(omega10*tt)
    x0_rec[i]=inverse_transform(T10[i],R10,x1_rec[i])

err_x1=float(np.max(np.abs(x1_rec-x1)))
err_x0=float(np.max(np.abs(x0_rec-x0)))
NESTED_INVERSE_C1_PASS=err_x1<1e-11
NESTED_INVERSE_C0_PASS=err_x0<1e-11
SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS=NESTED_INVERSE_C1_PASS and NESTED_INVERSE_C0_PASS
assert SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS

print("max_C1_inverse_error =",err_x1)
print("max_C0_inverse_error =",err_x0)
print("SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS =",SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS)

max_C1_inverse_error = 1.7763568394002505e-15
max_C0_inverse_error = 2.6645352591003757e-15
SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS = True


## 6 — Velocity decomposition and angular contribution

For

\[
\mathbf x_P=\mathbf T+R\mathbf x_C,
\]

differentiation gives

\[
\mathbf v_P
=
\dot{\mathbf T}
+
R\mathbf v_C
+
\dot R\mathbf x_C.
\]

Equivalently, with \(\Omega=\dot R R^T\),

\[
\dot R\mathbf x_C
=
\boldsymbol\omega\times(R\mathbf x_C).
\]

This angular term is absent from a static nested-cube picture.

In [7]:
v1=np.column_stack([
    -R_circle*omega10*np.sin(omega10*t),
     R_circle*omega10*np.cos(omega10*t),
     np.zeros_like(t)
])

v2_formula=np.empty_like(v1)
v2_exact=np.column_stack([
    -R_circle*(omega10+omega21)*np.sin((omega10+omega21)*t),
     R_circle*(omega10+omega21)*np.cos((omega10+omega21)*t),
     np.full_like(t,u2)
])

for i,tt in enumerate(t):
    R21=Rz(omega21*tt)
    rotated_x1=R21@x1[i]
    angular=np.cross(np.array([0.0,0.0,omega21]),rotated_x1)
    v2_formula[i]=np.array([0.0,0.0,u2]) + R21@v1[i] + angular

vel_err=float(np.max(np.abs(v2_formula-v2_exact)))
ANGULAR_VELOCITY_DECOMPOSITION_PASS=vel_err<1e-12
assert ANGULAR_VELOCITY_DECOMPOSITION_PASS

print("max_velocity_decomposition_error =",vel_err)
print("ANGULAR_VELOCITY_DECOMPOSITION_PASS =",ANGULAR_VELOCITY_DECOMPOSITION_PASS)

max_velocity_decomposition_error = 6.6058269965196814e-15
ANGULAR_VELOCITY_DECOMPOSITION_PASS = True


## 7 — Scientific classification

This notebook establishes only a kinematic possibility:

\[
\boxed{
\text{rectilinear}_{C_0}
\neq
\text{circular}_{C_1}
\neq
\text{helical}_{C_2}
}
\]

while inverse transforms recover the same local history.

Important locks:
- exact circle/helix morphology here is constructed through prescribed \(T(t)\) and \(R(t)\);
- generic nested angular motions produce more general curves;
- no force law explains \(\omega_{10}\), \(\omega_{21}\), or \(u_2\);
- no GVH-specific inter-scale angular dynamics is derived.

In [8]:
G272_TIME_DEPENDENT_NESTED_ROTATIONS_MATERIALIZED=True
G272_C0_RECTILINEAR_PASS=C0_RECTILINEAR_PASS
G272_C1_EXACT_CIRCLE_PASS=C1_EXACT_CIRCLE_PASS
G272_C2_EXACT_HELIX_PASS=C2_EXACT_HELIX_PASS
G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS=SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS
G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS=ANGULAR_VELOCITY_DECOMPOSITION_PASS

G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS=all([
    UPSTREAM_GATE,
    SO3_TIME_DEPENDENT_ROTATION_PASS,
    G272_C0_RECTILINEAR_PASS,
    G272_C1_EXACT_CIRCLE_PASS,
    G272_C2_EXACT_HELIX_PASS,
    G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS,
    G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS,
])

G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED=False
G272_OMEGA_LAW_DERIVED=False
G272_FORCE_LAW_DERIVED=False
G272_3D_TO_4D_EMERGENCE_DERIVED=False
G272_CORE_CHANGED=False
G272_NEW_GVH_PHYSICS_VALIDATED=False

G272_NEXT_AUTHORIZED=(
    "DERIVE-CANDIDATE-INTERSCALE-ANGULAR-LAW-OMEGA_k-AND-TEST-GENERIC-NONENGINEERED-TRAJECTORIES"
    if G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS
    else "REPAIR-NESTED-ANGULAR-CUBIC-MOTION-BENCHMARK"
)

assert G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS
assert not G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED
assert not G272_OMEGA_LAW_DERIVED
assert not G272_NEW_GVH_PHYSICS_VALIDATED

for name in [
    "G272_TIME_DEPENDENT_NESTED_ROTATIONS_MATERIALIZED",
    "G272_C0_RECTILINEAR_PASS",
    "G272_C1_EXACT_CIRCLE_PASS",
    "G272_C2_EXACT_HELIX_PASS",
    "G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS",
    "G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS",
    "G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS",
    "G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED",
    "G272_OMEGA_LAW_DERIVED",
    "G272_CORE_CHANGED",
    "G272_NEW_GVH_PHYSICS_VALIDATED",
]:
    print(name,"=",globals()[name])
print("G272_NEXT_AUTHORIZED =",G272_NEXT_AUTHORIZED)

G272_TIME_DEPENDENT_NESTED_ROTATIONS_MATERIALIZED = True
G272_C0_RECTILINEAR_PASS = True
G272_C1_EXACT_CIRCLE_PASS = True
G272_C2_EXACT_HELIX_PASS = True
G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS = True
G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS = True
G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS = True
G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED = False
G272_OMEGA_LAW_DERIVED = False
G272_CORE_CHANGED = False
G272_NEW_GVH_PHYSICS_VALIDATED = False
G272_NEXT_AUTHORIZED = DERIVE-CANDIDATE-INTERSCALE-ANGULAR-LAW-OMEGA_k-AND-TEST-GENERIC-NONENGINEERED-TRAJECTORIES


In [9]:
artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.27.2_Nested_Angular_Cubic_Motion_Line_Circle_Helix_Benchmark_FAST",
    "classification":"TOY_KINEMATIC_NESTED_FRAME_BENCHMARK",
    "upstream":UPSTREAM,
    "parameters":{
        "Tmax":Tmax,"N":N,"x00":x00,"v0":v0,"R_circle":R_circle,
        "omega10":omega10,"omega21":omega21,"u2":u2,
    },
    "numerical_witnesses":{
        "max_SO3_error":float(max(errs)),
        "max_C1_radius_error":float(np.max(np.abs(circle_radius-R_circle))),
        "fitted_C2_angular_rate":float(coef_phi[0]),
        "expected_C2_angular_rate":float(omega10+omega21),
        "max_C1_inverse_error":err_x1,
        "max_C0_inverse_error":err_x0,
        "max_velocity_decomposition_error":vel_err,
    },
    "flags":{
        "G272_TIME_DEPENDENT_NESTED_ROTATIONS_MATERIALIZED":G272_TIME_DEPENDENT_NESTED_ROTATIONS_MATERIALIZED,
        "G272_C0_RECTILINEAR_PASS":G272_C0_RECTILINEAR_PASS,
        "G272_C1_EXACT_CIRCLE_PASS":G272_C1_EXACT_CIRCLE_PASS,
        "G272_C2_EXACT_HELIX_PASS":G272_C2_EXACT_HELIX_PASS,
        "G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS":G272_SAME_KINEMATIC_HISTORY_RECONSTRUCTION_PASS,
        "G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS":G272_ANGULAR_VELOCITY_DECOMPOSITION_PASS,
        "G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS":G272_MINIMAL_ANGULAR_NESTED_BENCHMARK_PASS,
        "G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED":G272_INTERSCALE_ANGULAR_DYNAMICS_DERIVED,
        "G272_OMEGA_LAW_DERIVED":G272_OMEGA_LAW_DERIVED,
        "G272_FORCE_LAW_DERIVED":G272_FORCE_LAW_DERIVED,
        "G272_3D_TO_4D_EMERGENCE_DERIVED":G272_3D_TO_4D_EMERGENCE_DERIVED,
        "G272_CORE_CHANGED":G272_CORE_CHANGED,
        "G272_NEW_GVH_PHYSICS_VALIDATED":G272_NEW_GVH_PHYSICS_VALIDATED,
        "G272_NEXT_AUTHORIZED":G272_NEXT_AUTHORIZED,
    },
    "scope_note":"The line->circle->helix sequence is an engineered kinematic witness using prescribed translations and rotations. It demonstrates nested angular-frame consistency, not a derived physical inter-scale law."
}

export_dir=Path('/content/gvh_exports') if Path('/content').exists() else Path('/mnt/data')
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/'gvh_0.3.2.7.3.7.3.3.27.2_Nested_Angular_Cubic_Motion_Benchmark_FAST.json'
artifact_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False,default=lambda o:o.item() if isinstance(o,np.generic) else str(o)),encoding='utf-8')
print("artifact =",artifact_path)

artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.27.2_Nested_Angular_Cubic_Motion_Benchmark_FAST.json


### Provenance / scientific class

Recommended:

`AI-F + C + TOY-KINEMATIC-NESTED-ANGULAR-BENCHMARK`

Not `NOV-PASS`.

The structural lesson is:

\[
\boxed{
R_k=\mathrm{const}
\text{ is insufficient for the intended nested-cube picture; }
R_k(t),\ \omega_k(t)
\text{ must be allowed.}
}
\]

A future physical theory must derive the inter-scale angular laws instead of prescribing them.